# Learning Rate Scheduling: From Step Decay to One-Cycle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/learning_rate_scheduling.ipynb)

Implement linear, cosine, exponential, and one-cycle learning rate schedules from scratch. Includes an LR finder, early stopping, and a benchmark.

**Blog post:** [sesen.ai/blog/learning-rate-scheduling-step-decay-to-one-cycle](https://sesen.ai/blog/learning-rate-scheduling-step-decay-to-one-cycle)

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

## Four LR Schedules from Scratch

In [ ]:
def schedule_constant(start, end, pos):
    """Constant LR: ignores pos, always returns start."""
    return start

def schedule_linear(start, end, pos):
    """Linear decay: straight line from start to end."""
    return start + pos * (end - start)

def schedule_cosine(start, end, pos):
    """Cosine annealing: smooth S-curve from start to end."""
    return start + (1 + math.cos(math.pi * (1 - pos))) * (end - start) / 2

def schedule_exponential(start, end, pos):
    """Exponential decay: fast early drop, slow tail."""
    return start * (end / start) ** pos

# Visualise: pos goes from 0 (start of training) to 1 (end)
positions = np.linspace(0, 1, 100)
for name, fn in [("Constant", schedule_constant), ("Linear", schedule_linear),
                  ("Cosine", schedule_cosine), ("Exponential", schedule_exponential)]:
    lrs = [fn(0.1, 0.001, p) for p in positions]
    print(f"{name:12s}: start={lrs[0]:.4f}, mid={lrs[50]:.4f}, end={lrs[-1]:.4f}")

In [ ]:
# Plot all four schedules
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'Constant': '#6b7280', 'Linear': '#3b82f6', 'Cosine': '#22c55e', 'Exponential': '#f59e0b'}

for name, fn in [("Constant", schedule_constant), ("Linear", schedule_linear),
                  ("Cosine", schedule_cosine), ("Exponential", schedule_exponential)]:
    lrs = [fn(0.1, 0.001, p) for p in positions]
    ax.plot(positions * 100, lrs, linewidth=2, color=colors[name], label=name)

ax.set_xlabel('Training Progress (%)', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('Four Learning Rate Schedules (0.1 → 0.001)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## The 1-Cycle Policy

Warmup the LR for the first 30% of training, then decay for the remaining 70%.

In [ ]:
def combine_schedules(percentages, schedule_fns):
    """Combine multiple schedules over different phases of training.

    Args:
        percentages: fraction of training for each phase (must sum to 1)
        schedule_fns: list of (start, end, schedule_fn) for each phase
    """
    assert abs(sum(percentages) - 1.0) < 1e-6
    boundaries = np.cumsum([0] + list(percentages))

    def get_lr(pos):
        # Find which phase we're in
        for i in range(len(percentages)):
            if pos <= boundaries[i + 1] or i == len(percentages) - 1:
                # Rescale pos to [0, 1] within this phase
                phase_pos = (pos - boundaries[i]) / percentages[i]
                phase_pos = np.clip(phase_pos, 0, 1)
                start, end, fn = schedule_fns[i]
                return fn(start, end, phase_pos)
        return schedule_fns[-1][0]  # fallback

    return get_lr

# 1-cycle: warm up for 30%, then anneal for 70%
one_cycle = combine_schedules(
    [0.3, 0.7],
    [(0.01, 0.1, schedule_cosine),   # warm up: 0.01 -> 0.1
     (0.1, 0.001, schedule_cosine)]  # anneal:  0.1 -> 0.001
)

positions = np.linspace(0, 1, 100)
lrs = [one_cycle(p) for p in positions]
print(f"1-cycle: start={lrs[0]:.4f}, peak={max(lrs):.4f}, end={lrs[-1]:.4f}")

In [ ]:
# Plot 1-cycle schedule
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(positions * 100, lrs, linewidth=2.5, color='#8b5cf6')
ax.axvline(x=30, color='gray', linestyle='--', alpha=0.5, label='Warmup → Decay')
ax.set_xlabel('Training Progress (%)', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('1-Cycle Schedule: Warmup (30%) then Cosine Decay (70%)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## The Learning Rate Finder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = datasets.MNIST('data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)

def make_cnn(use_bn=True):
    layers = []
    channels = [(1, 8, 5), (8, 16, 3), (16, 32, 3), (32, 64, 3), (64, 64, 3)]
    for ni, nf, ks in channels:
        layers.append(nn.Conv2d(ni, nf, ks, padding=ks//2, stride=2, bias=not use_bn))
        if use_bn: layers.append(nn.BatchNorm2d(nf))
        layers.append(nn.ReLU())
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)]
    return nn.Sequential(*layers)

def lr_find(model, train_loader, min_lr=1e-5, max_lr=10, num_batches=200):
    """Exponentially increase LR over num_batches, record loss at each step."""
    optimiser = torch.optim.SGD(model.parameters(), lr=min_lr, momentum=0.9)
    lrs, losses = [], []
    best_loss = float('inf')
    mult = (max_lr / min_lr) ** (1 / num_batches)

    model.train()
    lr = min_lr
    batch_iter = iter(train_loader)
    for i in range(num_batches):
        try:
            images, labels = next(batch_iter)
        except StopIteration:
            batch_iter = iter(train_loader)
            images, labels = next(batch_iter)

        loss = F.cross_entropy(model(images), labels)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        lrs.append(lr)
        losses.append(loss.item())

        if loss.item() < best_loss:
            best_loss = loss.item()
        if loss.item() > best_loss * 10:
            break  # loss exploded, stop

        lr *= mult
        for pg in optimiser.param_groups:
            pg['lr'] = lr

    return lrs, losses

torch.manual_seed(42)
model = make_cnn()
lrs_found, losses_found = lr_find(model, train_loader)

In [ ]:
# Plot LR finder results
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lrs_found, losses_found, linewidth=1.5, color='#3b82f6')
ax.set_xscale('log')
ax.set_xlabel('Learning Rate (log scale)', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('LR Finder: Loss vs Learning Rate', fontsize=14)
ax.axvline(x=0.1, color='#ef4444', linestyle='--', alpha=0.7, label='Suggested peak LR ≈ 0.1')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Early Stopping

In [ ]:
class EarlyStopping:
    """Stop training when validation loss stops improving."""

    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0

    def should_stop(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

# Demo
es = EarlyStopping(patience=3)
val_losses = [0.5, 0.45, 0.43, 0.44, 0.44, 0.45]
for i, vl in enumerate(val_losses):
    stop = es.should_stop(vl)
    print(f"  Epoch {i+1}: val_loss={vl:.2f}, counter={es.counter}, stop={stop}")

## Benchmark: Constant vs Cosine vs 1-Cycle

In [ ]:
test_data = datasets.MNIST('data', train=False, transform=transform)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1000)

def train_with_schedule(schedule_fn, epochs=10, base_lr=0.1, seed=42):
    """Train CNN with a given LR schedule."""
    torch.manual_seed(seed)
    model = make_cnn()
    optimiser = torch.optim.SGD(model.parameters(), lr=base_lr, momentum=0.9)
    accs = []
    total_batches = epochs * len(train_loader)

    batch_count = 0
    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            # Update LR according to schedule
            pos = batch_count / total_batches
            lr = schedule_fn(pos)
            for pg in optimiser.param_groups:
                pg['lr'] = lr

            loss = F.cross_entropy(model(images), labels)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            batch_count += 1

        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in test_loader:
                correct += (model(images).argmax(1) == labels).sum().item()
        accs.append(correct / len(test_data) * 100)

    return accs

In [ ]:
# Three strategies
constant_fn = lambda pos: 0.01  # safe constant LR
cosine_fn = lambda pos: schedule_cosine(0.1, 0.001, pos)  # cosine from 0.1 to 0.001
one_cycle_fn = combine_schedules(
    [0.3, 0.7],
    [(0.01, 0.1, schedule_cosine), (0.1, 0.001, schedule_cosine)]
)

print("Constant LR=0.01 (10 epochs):")
acc_const = train_with_schedule(constant_fn, epochs=10)
print(f"  Final: {acc_const[-1]:.1f}%")

print("\nCosine 0.1->0.001 (10 epochs):")
acc_cos = train_with_schedule(cosine_fn, epochs=10)
print(f"  Final: {acc_cos[-1]:.1f}%")

print("\n1-cycle 0.01->0.1->0.001 (10 epochs):")
acc_1c = train_with_schedule(one_cycle_fn, epochs=10)
print(f"  Final: {acc_1c[-1]:.1f}%")

print("\nConstant LR=0.01 (20 epochs, baseline):")
acc_const20 = train_with_schedule(constant_fn, epochs=20)
print(f"  Final: {acc_const20[-1]:.1f}%")

In [ ]:
# Plot benchmark results
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(1, 11), acc_const, 'o-', color='#6b7280', linewidth=2, markersize=6,
        label=f'Constant lr=0.01 (10ep) → {acc_const[-1]:.1f}%')
ax.plot(range(1, 11), acc_cos, 'o-', color='#22c55e', linewidth=2, markersize=6,
        label=f'Cosine 0.1→0.001 (10ep) → {acc_cos[-1]:.1f}%')
ax.plot(range(1, 11), acc_1c, 'o-', color='#8b5cf6', linewidth=2, markersize=6,
        label=f'1-cycle (10ep) → {acc_1c[-1]:.1f}%')
ax.plot(range(1, 21), acc_const20, 's--', color='#6b7280', linewidth=1.5, markersize=4,
        alpha=0.6, label=f'Constant lr=0.01 (20ep) → {acc_const20[-1]:.1f}%')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('LR Schedule Benchmark on MNIST (5-layer CNN + BN)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(95, 100)
plt.tight_layout()
plt.show()

## Exercises

1. **Warm restarts** — Implement cosine annealing with warm restarts: after each cycle of `T` epochs, reset the LR back to the peak. Compare with plain cosine on a 30-epoch run.

2. **1-cycle warmup fraction** — Try 10%, 30%, and 50% warmup fraction for 1-cycle. Which gives the best final accuracy?

3. **LR finder for Adam** — Run the LR finder with `torch.optim.Adam` instead of SGD. How does the optimal LR range differ?

4. **Step decay** — Implement step decay (halve LR every 3 epochs) and add it to the benchmark. Where does it rank?

5. **Early stopping + 1-cycle** — Combine early stopping (patience=5) with the 1-cycle schedule on a 50-epoch run. At which epoch does it stop?

## References

- Smith, L.N. (2018). [Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates.](https://arxiv.org/abs/1708.07120)
- Smith, L.N. (2015). [Cyclical Learning Rates for Training Neural Networks.](https://arxiv.org/abs/1506.01186)
- Loshchilov, I. & Hutter, F. (2016). [SGDR: Stochastic Gradient Descent with Warm Restarts.](https://arxiv.org/abs/1608.03676)
- fast.ai course: [Practical Deep Learning for Coders, Lessons 9-10](https://course.fast.ai/).